## 1. Setup

Install from the repository root with `python -m pip install -r requirement.txt`, then select that environment as the notebook kernel. Restart the kernel after changing CAAF. This notebook imports the current repository sources and performs no file exports.

Preprocessing always uses the full demonstration data. The Ranking section provides a separate commented smoke-test call; skip the full ranking cell and uncomment the smoke-test cell when you want a short verification run.

Obtain the [channel-flow data from Figshare](https://doi.org/10.6084/m9.figshare.31896294) and place `V_data.mat`, `P_w.mat`, `x_edge.mat`, `y_edge.mat`, and `z_edge.mat` in `Wall-normal velocity prediction/`. `V_data.mat` is not included in this checkout. Missing files are reported explicitly; this notebook does not download or substitute data.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Locate the repository independently of the current notebook directory.
root = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "CAAF" / "__init__.py").is_file() and (p / "Demo").is_dir()), None)
if root is None:
    raise FileNotFoundError("Run this notebook from the CAAF repository or its Demo directory.")
sys.path.insert(0, str(root))
import CAAF

from Demo.utils.wall_normal_velocity_data_processing import load_wall_normal_velocity_data

## 2. Preprocessing

Read the velocity plane at wall-normal index 22 (approximately `y+ = 9.6` in the reference grid), with 19 streamwise and three spanwise probe locations. Each probe supplies a 19 × 19 pressure patch and its matching velocity history. Only the required velocity plane is read from the MATLAB 7.3 file.

All 57 trajectories are retained: 900 timesteps per trajectory yield 51,300 rows and 361 sensor candidates. Pressure columns use Fortran-order patch flattening, with streamwise position varying fastest. Sensor offsets use the respective x and z grid spacing and `Re_tau = 186`. Data remain in physical coordinates here; CAAF applies centered-max normalization.

In [ ]:
# Load the complete data and retain each probe's trajectory identifier.
X, y, sensor_coordinates, probe_ids = load_wall_normal_velocity_data(root / "Wall-normal velocity prediction")
assert X.shape[1] == 361 and y.shape == (len(X),)
assert len(np.unique(probe_ids)) == 57
assert np.isfinite(X).all() and np.isfinite(y).all()
print(f"Pressure shape: {X.shape}; velocity shape: {y.shape}; trajectories: {len(np.unique(probe_ids))}")

## 3. Ranking

The TCN uses two causal convolution layers with 64 channels and ten-step input histories to predict velocity at the last input timestep. Each probe history is split chronologically into 90% training and 10% validation before window construction. Training and validation windows never cross a trajectory or partition boundary. IG evaluates valid windows within each complete trajectory, including validation inputs.

The mean-pressure IG baseline becomes zero after centering. Sensor attribution is averaged over history steps. The built-in TCN uses causal padding, unlike the symmetric padding in the historical reference notebook.


The next cell runs full ranking. For a smoke test, skip it and uncomment every line of the following cell, then continue to the results table and plot. Smoke testing preserves all preprocessed data and clustering; it reduces only training and attribution effort. `verbose=True` shows each processing stage; set it to `False` to silence CAAF progress.

Normalization and clustering use all supplied inputs. Validation loss selects the model checkpoint and is not an independent predictive test score. These demos use current CAAF models and splitting rules rather than reproducing historical numerical rankings exactly.

In [ ]:
# Rank pressure sensors with ten-step TCN histories that stay within each probe.
sensor_indices, percentages = CAAF.rank_sensors(
    X, y, n_sensors=5, normalization="centered_max",
    clustering={"name": "ap", "preference": 0.9, "damping": 0.5,
                "max_iter": 10000, "convergence_iter": 50},
    model={"name": "tcn", "channels": (64, 64), "kernel_size": 3, "activation": "relu"},
    sequence={"length": 10, "horizon": 0, "groups": probe_ids},
    training={"epochs": 150, "n_runs": 5, "batch_size": 64,
              "optimizer": "adam", "lr": 1e-4, "weight_decay": 1e-5,
              "dtype": "float64", "scheduler": {"patience": 2, "eps": 1e-6}},
    ig={"baseline": "mean", "n_steps": 50, "max_samples": None,
        "aggregation": "mean_abs"},
    device="auto", verbose=True, return_percentages=True,
)

assert len(sensor_indices) == 5 and len(np.unique(sensor_indices)) == 5
assert np.isfinite(percentages).all() and (percentages >= 0).all()

In [ ]:
# # Rank pressure sensors with ten-step TCN histories that stay within each probe.
# sensor_indices, percentages = CAAF.rank_sensors(
#     X, y, n_sensors=5, normalization="centered_max",
#     clustering={"name": "ap", "preference": 0.9, "damping": 0.5,
#                 "max_iter": 10000, "convergence_iter": 50},
#     model={"name": "tcn", "channels": (64, 64), "kernel_size": 3, "activation": "relu"},
#     sequence={"length": 10, "horizon": 0, "groups": probe_ids},
#     training={"epochs": 2, "n_runs": 1, "batch_size": 64,
#               "optimizer": "adam", "lr": 1e-4, "weight_decay": 1e-5,
#               "dtype": "float64", "scheduler": {"patience": 2, "eps": 1e-6}},
#     ig={"baseline": "mean", "n_steps": 3, "max_samples": 512,
#         "aggregation": "mean_abs"},
#     device="auto", verbose=True, return_percentages=True,
# )
# 
# assert len(sensor_indices) == 5 and len(np.unique(sensor_indices)) == 5
# assert np.isfinite(percentages).all() and (percentages >= 0).all()

## 4. Results table

Indices are zero-based original sensor columns. Percentages retain their share of attribution across all cluster representatives, so selected values need not sum to 100%.

In [ ]:
# Align each sensor rank with its pressure-patch coordinates in wall units.
selected = sensor_coordinates[sensor_indices]
results = pd.DataFrame({"Rank": np.arange(1, len(sensor_indices) + 1),
                        "Sensor index": sensor_indices, "x+ offset": selected[:, 0],
                        "z+ offset": selected[:, 1], "Attribution (%)": percentages})
results.style.hide(axis="index").format({"x+ offset": "{:.3f}", "z+ offset": "{:.3f}", "Attribution (%)": "{:.3f}"})

## 5. Final sensor configuration

Sensor locations and rank labels below are computed from the selected ranking.

In [ ]:
# Plot the candidate pressure grid relative to the velocity target at the origin.
fig, ax = plt.subplots(figsize=(8, 7), layout="constrained")
ax.scatter(sensor_coordinates[:, 0], sensor_coordinates[:, 1], s=12, color="0.75", label="Pressure candidates")
selected = sensor_coordinates[sensor_indices]
ax.scatter(selected[:, 0], selected[:, 1], s=90, marker="x", color="tab:red", label="Selected sensors", zorder=4)
ax.scatter([0], [0], s=130, marker="*", facecolors="none", edgecolors="tab:blue", label="Velocity target", zorder=3)
for rank, (x_position, z_position) in enumerate(selected, start=1):
    ax.annotate(str(rank), (x_position, z_position), xytext=(7, 9 + 7 * ((rank - 1) % 2)),
                textcoords="offset points", arrowprops={"arrowstyle": "-", "color": "0.5"})
ax.axhline(0, color="0.6", linewidth=0.6, linestyle=":")
ax.axvline(0, color="0.6", linewidth=0.6, linestyle=":")
ax.set(xlabel="Streamwise offset x+", ylabel="Spanwise offset z+")
ax.set_aspect("equal")
ax.margins(0.15)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, 1.13), ncol=2)
plt.show()